# Unit 4.1 - Naïve Bayes Exercise
## GABRIEL DIANA & KEN MEIRO VILLAREAL- BSCS 3B

Naïve Bayes Implementation. 

## Dataset Setup

In [17]:
import numpy as np
import re
from collections import defaultdict, Counter
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

spam_messages = [
    "Free money now!!!",
    "Lowest price for your meds",
    "Win a free iPhone today",
    "Get 50% off, limited time!",
    "Click here for prizes!"
]

ham_messages = [
    "Hi mom, how are you?",
    "Are we still on for dinner?",
    "Let's catch up tomorrow at the office",
    "Meeting at 3 PM tomorrow",
    "Team meeting in the office",
    "Can you send the report?"
]

documents = spam_messages + ham_messages
labels = ['SPAM'] * len(spam_messages) + ['HAM'] * len(ham_messages)

test_sentences = [
    "Limited offer, click here!",
    "Meeting at 2 PM with the manager." ]

print(f"HAM messages: {len(ham_messages)}")

print(f"SPAM messages: {len(spam_messages)}")
print(f"Total documents: {len(documents)}")

HAM messages: 6
SPAM messages: 5
Total documents: 11


## Task 1: Manual Naïve Bayes Implementation

### a. Generate Bag of Words (Calculate Word Frequency)

In [18]:
def preprocess_text(text):
    text = text.lower()
    words = re.findall(r'\b[a-z]+\b', text)
    return words

def build_vocabulary(documents):
    vocabulary = set()
    for doc in documents:
        words = preprocess_text(doc)
        vocabulary.update(words)
    return sorted(vocabulary)

def generate_bag_of_words(documents, labels):
    vocabulary = build_vocabulary(documents)
    
    spam_word_counts = Counter()
    ham_word_counts = Counter()
    
    for doc, label in zip(documents, labels):
        words = preprocess_text(doc)
        if label == 'SPAM':
            spam_word_counts.update(words)
        else:
            ham_word_counts.update(words)
    
    return vocabulary, spam_word_counts, ham_word_counts

vocabulary, spam_word_counts, ham_word_counts = generate_bag_of_words(documents, labels)

print("Task 1a: BAG OF WORDS\n")
print(f"\nVocabulary size: {len(vocabulary)}")
print(f"Vocabulary: {vocabulary}\n")

print("SPAM Word Counts:")
for word, count in sorted(spam_word_counts.items()):
    print(f"  {word}: {count}")


Task 1a: BAG OF WORDS


Vocabulary size: 43
Vocabulary: ['a', 'are', 'at', 'can', 'catch', 'click', 'dinner', 'for', 'free', 'get', 'here', 'hi', 'how', 'in', 'iphone', 'let', 'limited', 'lowest', 'meds', 'meeting', 'mom', 'money', 'now', 'off', 'office', 'on', 'pm', 'price', 'prizes', 'report', 's', 'send', 'still', 'team', 'the', 'time', 'today', 'tomorrow', 'up', 'we', 'win', 'you', 'your']

SPAM Word Counts:
  a: 1
  click: 1
  for: 2
  free: 2
  get: 1
  here: 1
  iphone: 1
  limited: 1
  lowest: 1
  meds: 1
  money: 1
  now: 1
  off: 1
  price: 1
  prizes: 1
  time: 1
  today: 1
  win: 1
  your: 1


### b. Calculate Prior Probabilities

In [10]:
def calculate_prior_probabilities(labels):
    total_docs = len(labels)
    spam_count = labels.count('SPAM')
    ham_count = labels.count('HAM')
    
    prior_spam = spam_count / total_docs
    prior_ham = ham_count / total_docs
    
    return prior_spam, prior_ham, spam_count, ham_count

prior_spam, prior_ham, spam_count, ham_count = calculate_prior_probabilities(labels)

print("Task 1b: PRIOR PROBABILITIES")

print(f"\nTotal documents: {len(labels)}")
print(f"SPAM documents: {spam_count}")
print(f"HAM documents: {ham_count}")
print(f"\nP(SPAM) = {spam_count}/{len(labels)} = {prior_spam:.4f}")
print(f"P(HAM) = {ham_count}/{len(labels)} = {prior_ham:.4f}")

Task 1b: PRIOR PROBABILITIES

Total documents: 11
SPAM documents: 5
HAM documents: 6

P(SPAM) = 5/11 = 0.4545
P(HAM) = 6/11 = 0.5455


### c. Calculate Likelihood of Each Token (with Laplace Smoothing)

In [12]:
def calculate_likelihoods(vocabulary, spam_word_counts, ham_word_counts, alpha=1):
    vocab_size = len(vocabulary)
    total_spam_words = sum(spam_word_counts.values())
    total_ham_words = sum(ham_word_counts.values())
    
    spam_likelihoods = {}
    ham_likelihoods = {}
    
    for word in vocabulary:
        spam_likelihoods[word] = (spam_word_counts[word] + alpha) / (total_spam_words + alpha * vocab_size)
        ham_likelihoods[word] = (ham_word_counts[word] + alpha) / (total_ham_words + alpha * vocab_size)
    
    return spam_likelihoods, ham_likelihoods, total_spam_words, total_ham_words

spam_likelihoods, ham_likelihoods, total_spam_words, total_ham_words = calculate_likelihoods(
    vocabulary, spam_word_counts, ham_word_counts
)

print("TASK 1c: LIKELIHOOD CALCULATIONS")

print(f"\nTotal SPAM words: {total_spam_words}")
print(f"Total HAM words: {total_ham_words}")
print(f"Vocabulary size: {len(vocabulary)}")
print(f"\nUsing Laplace smoothing (alpha = 1)\n")

print("\nLikelihood Table:")
print(f"{'Word':<15} {'P(word|SPAM)':<20} {'P(word|HAM)':<20}")
print("=" * 60)

for word in vocabulary:
    print(f"{word:<15} {spam_likelihoods[word]:<20.6f} {ham_likelihoods[word]:<20.6f}")

TASK 1c: LIKELIHOOD CALCULATIONS

Total SPAM words: 21
Total HAM words: 33
Vocabulary size: 43

Using Laplace smoothing (alpha = 1)


Likelihood Table:
Word            P(word|SPAM)         P(word|HAM)         
a               0.031250             0.013158            
are             0.015625             0.039474            
at              0.015625             0.039474            
can             0.015625             0.026316            
catch           0.015625             0.026316            
click           0.031250             0.013158            
dinner          0.015625             0.026316            
for             0.046875             0.026316            
free            0.046875             0.013158            
get             0.031250             0.013158            
here            0.031250             0.013158            
hi              0.015625             0.026316            
how             0.015625             0.026316            
in              0.015625            

### d. Classify Test Sentences

In [21]:
def classify_naive_bayes(sentence, vocabulary, prior_spam, prior_ham, 
                         spam_likelihoods, ham_likelihoods, 
                         total_spam_words, total_ham_words, alpha=1):
    words = preprocess_text(sentence)
    vocab_size = len(vocabulary)
    
    log_prob_spam = np.log(prior_spam)
    log_prob_ham = np.log(prior_ham)
    
    print(f"\nClassifying: '{sentence}'")
    print(f"Words extracted: {words}\n")
    
    print("Calculation steps:")
    print(f"Initial log P(SPAM) = log({prior_spam:.4f}) = {log_prob_spam:.4f}")
    print(f"Initial log P(HAM) = log({prior_ham:.4f}) = {log_prob_ham:.4f}\n")
    
    for word in words:
        if word in vocabulary:
            log_prob_spam += np.log(spam_likelihoods[word])
            log_prob_ham += np.log(ham_likelihoods[word])
            print(f"  '{word}': P(word|SPAM)={spam_likelihoods[word]:.6f}, P(word|HAM)={ham_likelihoods[word]:.6f}")
        else:
            unknown_spam_prob = alpha / (total_spam_words + alpha * vocab_size)
            unknown_ham_prob = alpha / (total_ham_words + alpha * vocab_size)
            log_prob_spam += np.log(unknown_spam_prob)
            log_prob_ham += np.log(unknown_ham_prob)
            print(f"  '{word}': [UNKNOWN WORD] Using smoothing")
    
    print(f"\nFinal log P(SPAM|sentence) = {log_prob_spam:.4f}")
    print(f"Final log P(HAM|sentence) = {log_prob_ham:.4f}\n")
    
    if log_prob_spam > log_prob_ham:
        classification = 'SPAM'
        confidence = log_prob_spam - log_prob_ham
    else:
        classification = 'HAM'
        confidence = log_prob_ham - log_prob_spam
    
    print(f"Classification: {classification}")
    print(f"Confidence score: {confidence:.4f}")
    print("=" * 60)
    
    return classification, log_prob_spam, log_prob_ham

print("TASK 1d: CLASSIFY TEST SENTENCES (MANUAL IMPLEMENTATION)")

for test_sentence in test_sentences:
    classification, log_spam, log_ham = classify_naive_bayes(
        test_sentence, vocabulary, prior_spam, prior_ham,
        spam_likelihoods, ham_likelihoods,
        total_spam_words, total_ham_words
    )

TASK 1d: CLASSIFY TEST SENTENCES (MANUAL IMPLEMENTATION)

Classifying: 'Limited offer, click here!'
Words extracted: ['limited', 'offer', 'click', 'here']

Calculation steps:
Initial log P(SPAM) = log(0.4545) = -0.7885
Initial log P(HAM) = log(0.5455) = -0.6061

  'limited': P(word|SPAM)=0.031250, P(word|HAM)=0.013158
  'offer': [UNKNOWN WORD] Using smoothing
  'click': P(word|SPAM)=0.031250, P(word|HAM)=0.013158
  'here': P(word|SPAM)=0.031250, P(word|HAM)=0.013158

Final log P(SPAM|sentence) = -15.3445
Final log P(HAM|sentence) = -17.9291

Classification: SPAM
Confidence score: 2.5845

Classifying: 'Meeting at 2 PM with the manager.'
Words extracted: ['meeting', 'at', 'pm', 'with', 'the', 'manager']

Calculation steps:
Initial log P(SPAM) = log(0.4545) = -0.7885
Initial log P(HAM) = log(0.5455) = -0.6061

  'meeting': P(word|SPAM)=0.015625, P(word|HAM)=0.039474
  'at': P(word|SPAM)=0.015625, P(word|HAM)=0.039474
  'pm': P(word|SPAM)=0.015625, P(word|HAM)=0.026316
  'with': [UNKNOWN W

## Task 2: Using Scikit-Learn's Multinomial Naïve Bayes

### a. Train the Model

In [23]:
vectorizer = CountVectorizer()
X_train = vectorizer.fit_transform(documents)
y_train = labels

nb_classifier = MultinomialNB(alpha=1.0)
nb_classifier.fit(X_train, y_train)


print("TASK 2a: SCIKIT-LEARN MODEL TRAINING")

print("\nModel trained successfully!")
print(f"Feature names (vocabulary): {vectorizer.get_feature_names_out()}")
print(f"\nTraining accuracy: {nb_classifier.score(X_train, y_train) * 100:.2f}%")
print(f"\nClass prior probabilities:")
for i, class_label in enumerate(nb_classifier.classes_):
    print(f"  P({class_label}) = {np.exp(nb_classifier.class_log_prior_[i]):.4f}")

TASK 2a: SCIKIT-LEARN MODEL TRAINING

Model trained successfully!
Feature names (vocabulary): ['50' 'are' 'at' 'can' 'catch' 'click' 'dinner' 'for' 'free' 'get' 'here'
 'hi' 'how' 'in' 'iphone' 'let' 'limited' 'lowest' 'meds' 'meeting' 'mom'
 'money' 'now' 'off' 'office' 'on' 'pm' 'price' 'prizes' 'report' 'send'
 'still' 'team' 'the' 'time' 'today' 'tomorrow' 'up' 'we' 'win' 'you'
 'your']

Training accuracy: 100.00%

Class prior probabilities:
  P(HAM) = 0.5455
  P(SPAM) = 0.4545


### b. Classify Test Sentences

In [ ]:
X_test = vectorizer.transform(test_sentences)

predictions = nb_classifier.predict(X_test)
probabilities = nb_classifier.predict_proba(X_test)

print("TASK 2b: CLASSIFY TEST SENTENCES (SCIKIT-LEARN)")

for i, test_sentence in enumerate(test_sentences):
    print(f"\nTest sentence {i+1}: '{test_sentence}'")
    print(f"Classification: {predictions[i]}")
    print(f"Probabilities:")
    for j, class_label in enumerate(nb_classifier.classes_):
        print(f"  P({class_label}|sentence) = {probabilities[i][j]:.6f}")

TASK 2b: CLASSIFY TEST SENTENCES (SCIKIT-LEARN)

Test sentence 1: 'Limited offer, click here!'
Classification: SPAM
Probabilities:
  P(HAM|sentence) = 0.084717
  P(SPAM|sentence) = 0.915283

Test sentence 2: 'Meeting at 2 PM with the manager.'
Classification: HAM
Probabilities:
  P(HAM|sentence) = 0.978443
  P(SPAM|sentence) = 0.021557


## Conclusion

This exercise successfully implemented a Naïve Bayes classifier for spam detection using two approaches: a manual implementation from scratch and using scikit-learn's MultinomialNB. The manual implementation involved generating a Bag of Words representation, calculating prior probabilities and word likelihoods with Laplace smoothing, and classifying test sentences using log probabilities to avoid numerical underflow. Both implementations produced consistent results, correctly identifying spam messages containing promotional words and legitimate messages related to meetings and work, demonstrating the effectiveness of probabilistic text classification.